# 22 — Pull IMF World Economic Outlook (WEO) Forecasts

**Source:** IMF World Economic Outlook Database  
**Provider:** International Monetary Fund  
**Access:** IMF DataMapper API — free, no key required  
**Coverage:** ~190 economies, 1980–present + 5-year forward projections  
**Frequency:** Semi-annual releases (April and October); we pull the most recent release  

## What this notebook does
Pulls IMF WEO data via the DataMapper API. Crucially, WEO provides IMF *forward
projections* alongside historical estimates. Including the IMF's one-year-ahead
forecast as a feature at prediction time captures market-grade economic expectations
that lagged WDI actuals do not — useful for the 6-month prediction horizon.

## Features pulled

| Feature | WEO Code | Description |
|---|---|---|
| `gdp_growth_pct` | `NGDP_RPCH` | Real GDP growth (%) — actual + projected |
| `inflation_pct` | `PCPIPCH` | Consumer price inflation (%) |
| `fiscal_balance_gdp` | `GGXCNL_NGDP` | General govt net lending/borrowing (% GDP) |
| `public_debt_gdp` | `GGXWDN_NGDP` | General govt net debt (% GDP) |
| `current_account_gdp` | `BCA_NGDPD` | Current account balance (% GDP) |
| `unemployment_rate` | `LUR` | Unemployment rate (%) |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

## Note on forecasts vs. actuals
The `year` column in the output spans historical actuals and IMF projections — the
WEO does not distinguish them in the API response. A `year > current_year` flag
is added to mark forward projections explicitly.

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")
CURRENT_YEAR      = datetime.utcnow().year

WEO_API_BASE = "https://www.imf.org/external/datamapper/api/v1"

WEO_INDICATORS = {
    "NGDP_RPCH":    "gdp_growth_pct",
    "PCPIPCH":      "inflation_pct",
    "GGXCNL_NGDP":  "fiscal_balance_gdp",
    "GGXWDN_NGDP":  "public_debt_gdp",
    "BCA_NGDPD":    "current_account_gdp",
    "LUR":          "unemployment_rate",
}

print(f"Indicators  : {list(WEO_INDICATORS.keys())}")
print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/weo/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Pull WEO indicators

The DataMapper returns a nested JSON: indicator → country_code → {year: value}.
We flatten to long format, then pivot to wide country-year.

In [ ]:
def fetch_weo_indicator(code: str) -> pd.DataFrame:
    """Fetch one WEO indicator; return long-format DataFrame (iso3, year, value)."""
    url = f"{WEO_API_BASE}/{code}"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json().get("values", {}).get(code, {})

    rows = []
    for iso3, year_vals in data.items():
        for yr_str, val in year_vals.items():
            try:
                rows.append({"iso3": iso3, "year": int(yr_str), "value": float(val)})
            except (ValueError, TypeError):
                pass
    return pd.DataFrame(rows)


frames = {}
for code, col_name in WEO_INDICATORS.items():
    try:
        df_s = fetch_weo_indicator(code)
        df_s.rename(columns={"value": col_name}, inplace=True)
        frames[col_name] = df_s
        print(f"  {code:20s} → {col_name}: {len(df_s):,} obs")
    except Exception as e:
        print(f"  WARNING: could not pull {code}: {e}")

from functools import reduce
df_list = list(frames.values())
df_weo = reduce(lambda a, b: a.merge(b, on=["iso3", "year"], how="outer"), df_list)
df_weo = df_weo.sort_values(["iso3", "year"]).reset_index(drop=True)

# Flag forward projections
df_weo["is_projection"] = (df_weo["year"] > CURRENT_YEAR).astype(int)

print(f"\nFinal panel shape: {df_weo.shape}")
print(f"Countries   : {df_weo['iso3'].nunique()}")
print(f"Year range  : {df_weo['year'].min()}–{df_weo['year'].max()} "
      f"({df_weo[df_weo['is_projection']==1]['year'].nunique()} projection years)")
df_weo.head()

## Write to ADLS

In [ ]:
write_parquet(df_weo, f"raw/weo/{RUN_DATE}/weo_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("WEO pull complete")
print("=" * 55)
print(f"  Rows (country-years)  : {len(df_weo):,}")
print(f"  Countries             : {df_weo['iso3'].nunique()}")
print(f"  Historical years      : {df_weo[df_weo['is_projection']==0]['year'].nunique()}")
print(f"  Projection years      : {df_weo[df_weo['is_projection']==1]['year'].nunique()}")
print()
print("ADLS path written:")
print(f"  raw/weo/{RUN_DATE}/weo_panel.parquet")